# MLP Information Routing

How MLP layers route information: input-output mapping, feature amplification,
routing directions, cross-position routing, and summary.

## Why This Matters

MLP information routing provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_information_routing import (
    mlp_input_output_mapping, mlp_feature_amplification,
    mlp_routing_direction_analysis, mlp_cross_position_routing,
    mlp_routing_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Input-Output Mapping

How does MLP transform its input?

In [ ]:
for layer in range(4):
    result = mlp_input_output_mapping(model, tokens, layer=layer)
    print(f"  Layer {layer}: in={result['input_norm']:.4f}, out={result['output_norm']:.4f}, "
          f"amp={result['amplification']:.4f}, cos={result.get('input_output_cosine', 0):.4f}")

## Feature Amplification

Which neurons amplify or suppress their input most?

In [ ]:
for layer in range(4):
    result = mlp_feature_amplification(model, tokens, layer=layer, top_k=5)
    print(f"  Layer {layer}: mean_amp={result['mean_amplification']:.4f}")
    for n in result['per_neuron'][:3]:
        flag = ' [AMP]' if n['is_amplifying'] else ''
        print(f"    N{n['neuron']}: pre={n['pre_norm']:.4f}, "
              f"post={n['post_norm']:.4f}, ratio={n['amplification_ratio']:.4f}{flag}")

## Routing Directions

Principal directions of MLP output.

In [ ]:
for layer in range(4):
    result = mlp_routing_direction_analysis(model, tokens, layer=layer)
    print(f"  Layer {layer}: eff_rank={result['effective_rank']:.2f}")
    for d in result['directions'][:3]:
        bar = '█' * int(d['variance_explained'] * 30)
        print(f"    Dir {d['rank']}: sv={d['singular_value']:.4f}, "
              f"var={d['variance_explained']:.2%} {bar}")

## Cross-Position Routing

Does MLP treat positions similarly or differently?

In [ ]:
for layer in range(4):
    result = mlp_cross_position_routing(model, tokens, layer=layer)
    flag = ' [POS-SPECIFIC]' if result['is_position_specific'] else ''
    print(f"  Layer {layer}: sim={result['mean_similarity']:.4f}{flag}")

## Routing Summary

Cross-layer overview of MLP information routing.

In [ ]:
result = mlp_routing_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: amp={p['amplification']:.4f}, "
          f"cos={p['input_output_cosine']:.4f}, "
          f"pos_sim={p['position_similarity']:.4f}, "
          f"rank={p['effective_rank']:.2f}")